In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

## Dataset

Total images: 9540

*   Train: 6,678
*   Test: 1,431
*   Val: 1,431



In [ ]:
import kagglehub
import os

# Download dataset
PATH = kagglehub.dataset_download("ismailfaridi/glaucoma-dataset")

TRAIN_DIR = os.path.join(PATH, "train")
VAL_DIR   = os.path.join(PATH, "val")
TEST_DIR  = os.path.join(PATH, "test")

print("Train directory:", TRAIN_DIR)
print("Validation directory:", VAL_DIR)
print("Test directory:", TEST_DIR)

100%|██████████| 524M/524M [00:13<00:00, 41.7MB/s]

Extracting files...


Train directory: /root/.cache/kagglehub/datasets/ismailfaridi/glaucoma-dataset/versions/1/train
Validation directory: /root/.cache/kagglehub/datasets/ismailfaridi/glaucoma-dataset/versions/1/val
Test directory: /root/.cache/kagglehub/datasets/ismailfaridi/glaucoma-dataset/versions/1/test


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
import os
import tensorflow as tf

# Adjust paths to include the nested 'Glaucoma_Dataset' folder
TRAIN_DIR = os.path.join(PATH, "Glaucoma_Dataset", "train")
VAL_DIR   = os.path.join(PATH, "Glaucoma_Dataset", "val")
TEST_DIR  = os.path.join(PATH, "Glaucoma_Dataset", "test")

print("Loading datasets...")
train_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    seed = 123,             # For reproducibility
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    label_mode = "binary"
)

val_data = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    seed = 123,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    label_mode = "binary"
)

test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    label_mode = "binary",
    shuffle = False         # Important for consistent metric calculation
)

Loading datasets...
Found 6678 files belonging to 2 classes.
Found 1431 files belonging to 2 classes.
Found 1431 files belonging to 2 classes.


In [ ]:
class_names = train_data.class_names
print(f"Class names: {class_names}")

Class names: ['Glaucoma_Negative', 'Glaucoma_Positive']


In [ ]:
base_model = EfficientNetB0(
    include_top = False,
    weights = "imagenet",
    input_shape = (*IMG_SIZE, 3)
)

base_model.trainable = False


# CODE REMAINING


model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

model.summary()

In [ ]:
early_stopping = EarlyStopping(
    monitor = "val_loss",
    patience = 5,
    restore_best_weights = True,
    verbose = 1
)

history = model.fit(
    train_data,
    validation_data = val_data,
    epochs = 30,
    callbacks = [early_stopping]
)

In [ ]:
loss, acc = model.evaluate(test_data)
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {acc:.4f}")

In [ ]:
# Save the model
from google.colab import drive
drive.mount('/content/drive')

MODEL_SAVE_PATH = "/content/drive/MyDrive/glaucoma_detector_EfficientNetB0_model.h5"
model.save(MODEL_SAVE_PATH)
print(f"Model successfully saved to {MODEL_SAVE_PATH}.")